<a href="https://colab.research.google.com/github/Moquiuti/LangGraph_Orquestrando_agentes_e_multiagentes/blob/main/Agente_React.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U langchain
!pip install -q -U langchain-google-genai
!pip install -q -U langgraph
!pip install -q -U python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.1/114.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.3/234.3 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 3.9 MB/s eta 0:00:00


In [2]:
import os
import re
from typing import Dict, Callable, Optional

from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# Importando LangGraph apenas para já validar o ambiente do curso
from langgraph.graph import StateGraph

In [3]:
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    print("API Key carregada pelos Secrets do Colab.")
except Exception:
    load_dotenv()
    print("Tentando carregar API Key via arquivo .env.")

API Key carregada pelos Secrets do Colab.


In [4]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

resposta = llm.invoke("Responda apenas: Hello World")
print(resposta.content)

Hello World


Criando o inventário - irei simular uma base de produtos de informática.

In [5]:
inventario = {
    "notebook": {
        "estoque": 8,
        "preco": 4500.00
    },
    "mouse": {
        "estoque": 35,
        "preco": 89.90
    },
    "teclado": {
        "estoque": 20,
        "preco": 179.90
    },
    "monitor": {
        "estoque": 12,
        "preco": 1199.90
    },
    "headset": {
        "estoque": 15,
        "preco": 249.90
    },
    "webcam": {
        "estoque": 5,
        "preco": 329.90
    }
}

In [6]:
def consultar_estoque(produto: str) -> str:
    produto = produto.strip().lower()

    if produto not in inventario:
        return f"Produto '{produto}' não encontrado no inventário."

    quantidade = inventario[produto]["estoque"]
    return f"O produto '{produto}' possui {quantidade} unidades em estoque."

In [7]:
def consultar_preco_produto(produto: str) -> str:
    produto = produto.strip().lower()

    if produto not in inventario:
        return f"Produto '{produto}' não encontrado no inventário."

    preco = inventario[produto]["preco"]
    return f"O produto '{produto}' custa R$ {preco:.2f}."

In [8]:
def identificar_produto_mais_caro(_: str = "") -> str:
    produto_mais_caro = max(
        inventario.items(),
        key=lambda item: item[1]["preco"]
    )

    nome = produto_mais_caro[0]
    preco = produto_mais_caro[1]["preco"]

    return f"O produto mais caro do inventário é '{nome}', custando R$ {preco:.2f}."

In [9]:
def calcular_valor_total_lista(lista_produtos: str) -> str:
    if not lista_produtos or not lista_produtos.strip():
        return "Informe uma lista no formato: produto:quantidade, produto:quantidade."

    itens = lista_produtos.split(",")
    total = 0
    detalhes = []
    erros = []

    for item in itens:
        partes = item.strip().split(":")

        if len(partes) != 2:
            erros.append(f"Item inválido: '{item.strip()}'. Use o formato produto:quantidade.")
            continue

        produto = partes[0].strip().lower()

        try:
            quantidade = int(partes[1].strip())
        except ValueError:
            erros.append(f"Quantidade inválida para '{produto}'.")
            continue

        if produto not in inventario:
            erros.append(f"Produto '{produto}' não encontrado no inventário.")
            continue

        preco = inventario[produto]["preco"]
        subtotal = preco * quantidade
        total += subtotal

        detalhes.append(
            f"{produto}: {quantidade} x R$ {preco:.2f} = R$ {subtotal:.2f}"
        )

    if not detalhes:
        return "Não foi possível calcular o total. " + " ".join(erros)

    resposta = "Resumo do cálculo:\n"
    resposta += "\n".join(detalhes)
    resposta += f"\n\nValor total: R$ {total:.2f}"

    if erros:
        resposta += "\n\nObservações:\n" + "\n".join(erros)

    return resposta

In [10]:
ferramentas = {
    "consultar_estoque": consultar_estoque,
    "consultar_preco_produto": consultar_preco_produto,
    "identificar_produto_mais_caro": identificar_produto_mais_caro,
    "calcular_valor_total_lista": calcular_valor_total_lista
}

Prompt ReAct - Aqui está o coração da aula.

In [11]:
prompt_sistema = """
Você é um agente ReAct especializado em gerenciamento de inventário de produtos de informática.

Você pode raciocinar e usar ferramentas para responder perguntas do usuário.

Use sempre este formato quando precisar executar uma ferramenta:

Pensamento: explique brevemente o que você precisa descobrir.
Ação: nome_da_ferramenta: entrada_para_a_ferramenta
Pausa

Depois que receber uma Observação, continue o raciocínio.

Quando tiver a resposta final, use:

Resposta: sua resposta final para o usuário.

Ferramentas disponíveis:

1. consultar_estoque
Descrição: consulta a quantidade em estoque de um produto.
Entrada: nome do produto.
Exemplo:
Ação: consultar_estoque: notebook

2. consultar_preco_produto
Descrição: consulta o preço de um produto.
Entrada: nome do produto.
Exemplo:
Ação: consultar_preco_produto: mouse

3. identificar_produto_mais_caro
Descrição: identifica o produto mais caro do inventário.
Entrada: deixe vazio ou escreva inventário.
Exemplo:
Ação: identificar_produto_mais_caro: inventário

4. calcular_valor_total_lista
Descrição: calcula o valor total de uma lista de produtos.
Entrada: lista no formato produto:quantidade, produto:quantidade.
Exemplo:
Ação: calcular_valor_total_lista: notebook:2, mouse:3

Produtos disponíveis no inventário:
- notebook
- mouse
- teclado
- monitor
- headset
- webcam

Regras importantes:
- Não invente produtos.
- Não invente preços.
- Não invente estoque.
- Sempre que precisar de preço ou estoque, use uma ferramenta.
- Se o produto não existir, informe ao usuário com clareza.
"""

In [12]:
class Agent:
    def __init__(
        self,
        sistema: str,
        ferramentas: Dict[str, Callable[[str], str]],
        modelo: ChatGoogleGenerativeAI
    ):
        self.sistema = sistema
        self.ferramentas = ferramentas
        self.modelo = modelo
        self.historico = []

    def call(self, mensagem_usuario: str) -> str:
        mensagens = [SystemMessage(content=self.sistema)]
        mensagens.extend(self.historico)
        mensagens.append(HumanMessage(content=mensagem_usuario))

        resposta = self.modelo.invoke(mensagens)
        conteudo = resposta.content

        self.historico.append(HumanMessage(content=mensagem_usuario))
        self.historico.append(AIMessage(content=conteudo))

        return conteudo

    def executar_ferramenta(self, nome_ferramenta: str, entrada: str) -> str:
        nome_ferramenta = nome_ferramenta.strip()
        entrada = entrada.strip()

        if nome_ferramenta not in self.ferramentas:
            return f"Ferramenta '{nome_ferramenta}' não encontrada."

        try:
            return self.ferramentas[nome_ferramenta](entrada)
        except Exception as erro:
            return f"Erro ao executar a ferramenta '{nome_ferramenta}': {str(erro)}"

    def extrair_acao(self, texto: str) -> Optional[tuple]:
        padrao = r"Ação:\s*([a-zA-Z_]+)\s*:\s*(.*)"
        resultado = re.search(padrao, texto)

        if not resultado:
            return None

        nome_ferramenta = resultado.group(1).strip()
        entrada = resultado.group(2).strip()

        return nome_ferramenta, entrada

    def executar(self, pergunta: str, max_iteracoes: int = 5) -> str:
        resposta = self.call(pergunta)

        for _ in range(max_iteracoes):
            print("\n--- RESPOSTA DO AGENTE ---")
            print(resposta)

            acao = self.extrair_acao(resposta)

            if not acao:
                return resposta

            nome_ferramenta, entrada = acao

            observacao = self.executar_ferramenta(nome_ferramenta, entrada)

            print("\n--- OBSERVAÇÃO ---")
            print(observacao)

            resposta = self.call(f"Observação: {observacao}")

        return "O agente atingiu o limite máximo de iterações sem concluir a resposta."

In [13]:
def RunReactAgent(pergunta: str) -> str:
    modelo = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0
    )

    agente = Agent(
        sistema=prompt_sistema,
        ferramentas=ferramentas,
        modelo=modelo
    )

    resposta_final = agente.executar(pergunta)
    return resposta_final

Testando o agente

In [14]:
#Consultar estoque
resposta = RunReactAgent("Quantos notebooks temos em estoque?")
print("\n=== RESPOSTA FINAL ===")
print(resposta)


--- RESPOSTA DO AGENTE ---
Pensamento: O usuário quer saber a quantidade de notebooks em estoque. Preciso usar a ferramenta `consultar_estoque` para obter essa informação.
Ação: consultar_estoque: notebook
Pausa

--- OBSERVAÇÃO ---
O produto 'notebook' possui 8 unidades em estoque.

--- RESPOSTA DO AGENTE ---
Resposta: Temos 8 notebooks em estoque.

=== RESPOSTA FINAL ===
Resposta: Temos 8 notebooks em estoque.


In [15]:
#Consultar preço
resposta = RunReactAgent("Qual é o preço do monitor?")
print("\n=== RESPOSTA FINAL ===")
print(resposta)


--- RESPOSTA DO AGENTE ---
Pensamento: O usuário quer saber o preço do monitor. Preciso usar a ferramenta `consultar_preco_produto` para obter essa informação.
Ação: consultar_preco_produto: monitor
Pausa

--- OBSERVAÇÃO ---
O produto 'monitor' custa R$ 1199.90.

--- RESPOSTA DO AGENTE ---
Resposta: O preço do monitor é R$ 1199.90.

=== RESPOSTA FINAL ===
Resposta: O preço do monitor é R$ 1199.90.


In [16]:
#Produto inexistente
resposta = RunReactAgent("Qual é o preço do tablet?")
print("\n=== RESPOSTA FINAL ===")
print(resposta)


--- RESPOSTA DO AGENTE ---
O produto "tablet" não está disponível no inventário.

=== RESPOSTA FINAL ===
O produto "tablet" não está disponível no inventário.


In [17]:
#Produto mais caro
resposta = RunReactAgent("Qual é o produto mais caro do inventário?")
print("\n=== RESPOSTA FINAL ===")
print(resposta)


--- RESPOSTA DO AGENTE ---
Pensamento: O usuário quer saber qual é o produto mais caro. A ferramenta `identificar_produto_mais_caro` pode fornecer essa informação.
Ação: identificar_produto_mais_caro: inventário
Pausa

--- OBSERVAÇÃO ---
O produto mais caro do inventário é 'notebook', custando R$ 4500.00.

--- RESPOSTA DO AGENTE ---
Resposta: O produto mais caro do inventário é o notebook, custando R$ 4500.00.

=== RESPOSTA FINAL ===
Resposta: O produto mais caro do inventário é o notebook, custando R$ 4500.00.


In [18]:
#Calcular valor total
resposta = RunReactAgent("Calcule o valor total de 2 notebooks, 3 mouses e 1 teclado.")
print("\n=== RESPOSTA FINAL ===")
print(resposta)


--- RESPOSTA DO AGENTE ---
Ação: calcular_valor_total_lista: notebook:2, mouse:3, teclado:1
Pausa

--- OBSERVAÇÃO ---
Resumo do cálculo:
notebook: 2 x R$ 4500.00 = R$ 9000.00
mouse: 3 x R$ 89.90 = R$ 269.70
teclado: 1 x R$ 179.90 = R$ 179.90

Valor total: R$ 9449.60

--- RESPOSTA DO AGENTE ---
Resposta: O valor total de 2 notebooks, 3 mouses e 1 teclado é R$ 9449.60.

=== RESPOSTA FINAL ===
Resposta: O valor total de 2 notebooks, 3 mouses e 1 teclado é R$ 9449.60.


In [19]:
#Loop interativo para simular conversas
modelo = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

agente = Agent(
    sistema=prompt_sistema,
    ferramentas=ferramentas,
    modelo=modelo
)

print("Agente ReAct iniciado.")
print("Digite 'sair' para encerrar.\n")

while True:
    pergunta = input("Você: ")

    if pergunta.lower() in ["sair", "exit", "quit"]:
        print("Conversa encerrada.")
        break

    resposta = agente.executar(pergunta)

    print("\nAgente:")
    print(resposta)
    print("\n" + "-" * 80 + "\n")

Agente ReAct iniciado.
Digite 'sair' para encerrar.

Você: Me recomenda dois produtos

--- RESPOSTA DO AGENTE ---
Claro! Posso te recomendar o **notebook** e o **mouse**.

Se você quiser saber mais sobre o estoque ou preço desses produtos, ou tiver alguma preferência específica, me diga!

Agente:
Claro! Posso te recomendar o **notebook** e o **mouse**.

Se você quiser saber mais sobre o estoque ou preço desses produtos, ou tiver alguma preferência específica, me diga!

--------------------------------------------------------------------------------

Você: sair
Conversa encerrada.
